In [ ]:
import os
import numpy as np
import copy as cp
import json
from ase.io import read, write
#import calorine
from calorine.nep import setup_training, get_descriptors
import phonopy
from hiphive.structure_generation import generate_phonon_rattled_structures
import matplotlib.pyplot as plt
from src.latexfig import LatexFigure
#from src.plotsettings import PlotSettings
#PlotSettings().set_global_style()


In [ ]:
#fig, ax = latex_figure(aspect_ratio=0.7, nrows=3, ncols=3)
#ax[0].plot([0, 1], [0, 1])
#ax[1].plot([0, 1], [0, 1])
#plt.show()

In [ ]:


def create_scaled_figure_with_labels(nrows=2, ncols=1, textwidth_pt=369, scale=0.8,
                                     figsize_ratio=0.6, hspace=0.5, colors=None):
    """
    Create a figure scaled to LaTeX text width, with colored boxes around axes including labels and an outer box.
    """
    pt_to_inch = 1 / 72.27
    fig_width_in = textwidth_pt * pt_to_inch
    fig_height_in = fig_width_in * figsize_ratio

    fig, axes = plt.subplots(nrows, ncols, figsize=(fig_width_in, fig_height_in))
    if nrows * ncols == 1:
        axes = [axes]

    fig.subplots_adjust(hspace=hspace, left=0.1, right=0.9, top=0.9, bottom=0.1)

    fig.patch.set_linewidth(2)
    fig.patch.set_edgecolor('cornflowerblue')

    if colors is None:
        colors = ['red', 'green', 'orange', 'purple', 'brown']

    fig.canvas.draw()  # important to render text

    renderer = fig.canvas.get_renderer()

    for ii, ax in enumerate(axes):
        ax.set_title(f'Title {ii+1}')
        ax.set_xlabel(f'X-Label {ii+1}')
        ax.set_ylabel(f'Y-Label {ii+1}')

        fig.canvas.draw()  # update renderer

        # Get the tight bbox including title, labels, ticks
        bbox = ax.get_tightbbox(renderer)
        x0, y0, width, height = bbox.transformed(fig.transFigure.inverted()).bounds

        # Scale the tight bbox
        x_center = x0 + width / 2
        y_center = y0 + height / 2
        width_scaled = width * scale
        height_scaled = height * scale
        x0_scaled = x_center - width_scaled / 2
        y0_scaled = y_center - height_scaled / 2

        # Draw the scaled rectangle around the axes including labels
        fig.add_artist(plt.Rectangle(
            (x0_scaled, y0_scaled), width_scaled, height_scaled,
            edgecolor=colors[ii % len(colors)], linewidth=1.5, fill=False
        ))

        # Optional: shift the axes itself slightly to match scaled bbox
        ax_pos = ax.get_position()
        ax_width_scaled = ax_pos.width * scale
        ax_height_scaled = ax_pos.height * scale
        ax.set_position([
            ax_pos.x0 + (ax_pos.width - ax_width_scaled)/2,
            ax_pos.y0 + (ax_pos.height - ax_height_scaled)/2,
            ax_width_scaled,
            ax_height_scaled
        ])

    return fig, axes

In [ ]:
fig, axes = create_scaled_figure_with_labels(
    nrows=2,
    textwidth_pt=369,  # LaTeX text width in pt
    scale=0.6,         # inner boxes 80% size
    figsize_ratio=0.62, # height/width ratio
    hspace=0.6
)

fig.savefig('scaled_figuressss.pdf')  # outer blue box matches text width
plt.show()

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1)

# add a bit more breathing room around the axes for the frames
#fig.subplots_adjust(top=0.85, bottom=0.15, left=0.2, hspace=0.8)

fig.patch.set_linewidth(2)
fig.patch.set_edgecolor('cornflowerblue')

plt.ion()
fig.canvas.draw()

colors = ['red', 'green', 'blue']

for ii, ax in enumerate(axes):
    ax.set_title(f'Title {ii+1}')
    ax.set_ylabel(f'Y-Label {ii+1}')
    ax.set_xlabel(f'X-Label {ii+1}')
    bbox = ax.get_tightbbox(fig.canvas.get_renderer())
    x0, y0, width, height = bbox.transformed(fig.transFigure.inverted()).bounds
    # slightly increase the very tight bounds:
    xpad = 0.0 * width
    ypad = 0.0 * height
    fig.add_artist(plt.Rectangle((x0-xpad, y0-ypad), width+2*xpad, height+2*ypad, edgecolor=colors[ii], linewidth=1, fill=False))

fig.savefig('test2.pdf', bbox_inches='tight')
plt.show()

## Data pre-processing

In [ ]:
# Load dictionary from .json file
with open(os.path.join('MLtest', 'dft_params.json'), 'r') as f:
    dft_params = json.load(f)
dft_params

In [ ]:
from src.activeNEP import ActiveLearningNEP
NEP = ActiveLearningNEP('results/nep')

In [ ]:
# 0. Initial data generation
NEP.prepare_dataset()

In [ ]:
# 1. DFT labeling (SIESTA)
#NEP.run_DFT()      # run on HPC CPU node (sylg.fysik.dtu.dk)

In [ ]:
# 2. Train NEP model (nep.txt)
params = dict(cutoff=[8, 4],
              neuron=30,
              generation=100000,
              batch=1000000)
NEP.setup_nep(params)
#NEP.train_nep()    # run on HPC GPU node (surt.fysik.dtu.dk or sara.fysik.dtu.dk)

In [ ]:
#3. Extract descriptors (NEP prediction mode)
#NEP.run_prediction_mode()  # run on HPC GPU node (surt.fysik.dtu.dk or sara.fysik.dtu.dk)
NEP.compute_descriptors()


In [ ]:
calorine_desc = NEP._calculate_descriptors(NEP.train_data)

In [ ]:
NEP.descriptors - calorine_desc

In [ ]:
#4. Build Active Set (.asi) with MaxVol
NEP.build_active_set()


In [ ]:
from ase.io import read
new_structures = read('results/nep/train.xyz', index=':')


In [ ]:
from src.structure import Perovskite
atoms = Perovskite('BaTiO3').atoms

In [ ]:
from src.structure import get_reduced_formula
get_reduced_formula(new_structures[-1])

In [ ]:
def check_if_bulk(atoms):
    pbc = atoms.get_pbc()
    if not all(pbc):
        return False
    return True

In [ ]:
from ase.build import sort

In [ ]:
get_reduced_formula(atoms)

In [ ]:
get_reduced_formula(new_structures[0])

In [ ]:
new_structures[0].get_chemical_symbols()

In [ ]:
check_if_bulk(atoms)

In [ ]:
get_kspacing_from_kgrid(new_structures[-1], (10, 10, 10))

In [ ]:
def get_kspacing_from_kgrid(atoms, kgrid):
    cell = atoms.get_cell()
    rec = 2 * np.pi * np.linalg.inv(cell).T
    b_lengths = np.linalg.norm(rec, axis=1)

    kgrid = np.array(kgrid)
    kspacing = b_lengths / kgrid

    return kspacing

def kgrid_from_spacing(atoms, kspacing):
    cell = atoms.get_cell()
    rec = 2 * np.pi * np.linalg.inv(cell).T
    b_lengths = np.linalg.norm(rec, axis=1)
    kpts = np.maximum(1, np.round(b_lengths / kspacing).astype(int))
    return kpts

In [ ]:
new_structures[-1]

In [ ]:
import numpy as np

#atoms = new_structures[0]
kspacing=0.13

# Get cell (in Å)
cell = atoms.get_cell()

# Reciprocal lattice (ASE gives 2π * reciprocal vectors)
rec = 2 * np.pi * np.linalg.inv(cell).T

# Lengths of reciprocal vectors
b_lengths = np.linalg.norm(rec, axis=1)

# Number of k-points along each direction
kpts = np.maximum(1, np.round(b_lengths / kspacing).astype(int))



In [ ]:
kpts

In [ ]:
new_structures[0]

In [ ]:
new_structures[0].pbc

In [ ]:
50*(1/np.linalg.norm(new_structures[0].cell[:], axis=1))

In [ ]:
NEP.filter_structures(new_structures, gamma_th=1)

In [ ]:
force_errors = []
gammas = []
from calorine.calculators import CPUNEP

for struct in new_structures:
    try:
        forces_DFT = struct.get_forces().copy()
    except RuntimeError:
        continue
    calculator = CPUNEP('results/MLtest/iteration_1/nep.txt')
    struct.calc = calculator
    forces_ML = struct.get_forces()
    # Calculate maximum force error per atom in the structure
    force_error = np.max(np.abs(forces_DFT - forces_ML), axis=1)
    gamma = struct.arrays["gamma"]
    force_errors.append(force_error)
    gammas.append(gamma)


In [ ]:

# Plot force error vs gamma
lf = LatexFigure()
fig, axes = lf.create()

gammas_max = [np.max(gammas[i]) for i in range(len(gammas))]
forces_max = [np.max(force_errors[i]) for i in range(len(force_errors))]

for ax in axes:
    ax.scatter(gammas_max, forces_max, alpha=0.5)
    ax.set_xlabel('$\\gamma$')
    ax.set_ylabel('|dF| (eV/Å)')
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlim(1e-1, 1e2)
    ax.set_ylim(1e-3, 1e1)

# Make it log-log plot

plt.show()

In [ ]:
import phonopy as ph
from src.phononcalc import phonon_to_atoms
phonon = ph.load('results/MLtest/BaTiO3.yaml')

# Produce the force constants
phonon.produce_force_constants()
# Get the force constants matrix
fc = phonon.force_constants

# Convert the phonon object to an ASE Atoms object
atoms = phonon_to_atoms(phonon, cell='super')
new_structures = NEP._get_rattled_structures(atoms, fc, 1000, 1000)

In [ ]:
new_structures = [NEP.train_data[i].repeat((2, 2, 2)) for i in range(len(NEP.train_data))]

In [ ]:
from ase.visualize import view
view(new_structures)

In [ ]:
NEP.filter_structures(new_structures, gamma_th=2)

In [ ]:
filtered_structures = read('results/MLtest/iteration_1/large_gamma.xyz', index=':')

In [ ]:
NEP.select_structures(filtered_structures)

In [ ]:
selected_structures = read('results/MLtest/iteration_1/newdata.xyz', index=':')

In [ ]:
selected_structures[0].arrays["gamma"]

In [ ]:
np.array(selected_structures[0].get_chemical_symbols())

In [ ]:
view(new_structures[0])

In [ ]:
view(new_structures)

In [ ]:
view(new_structures[0])

In [ ]:
import numpy as np
#gamma_model = np.array([max(NEP.train_data[i].arrays["gamma"]) for i in range(len(NEP.train_data))])
#gamma_new_structures = np.array([max(new_structures[i].arrays["gamma"]) for i in range(len(new_structures))])
gamma_filtered_structures = np.array([max(filtered_structures[i].arrays["gamma"]) for i in range(len(filtered_structures))])
gamma_selected_structures = np.array([max(selected_structures[i].arrays["gamma"]) for i in range(len(selected_structures))])
gamma_min = min(gamma_filtered_structures.min(), gamma_selected_structures.min())
gamma_max = max(gamma_filtered_structures.max(), gamma_selected_structures.max())
print(f'Min extrapolation grade across structures: {gamma_min}')
print(f'Max extrapolation grade across structures: {gamma_max}')

In [ ]:
# Plot distribution of gamma
import matplotlib.pyplot as plt
import math
lf = LatexFigure()

g_min = math.floor(gamma_min * 2) / 2
g_max = math.ceil(gamma_max * 2) / 2

fig, axes = lf.create(AR=0.6)

# Plot vertical line at gamma = 2
axes[0].axvline(x=2, color='black', linestyle='--', label='$\\gamma = 2$')

axes[0].hist(gamma_filtered_structures, label='Filtered Structures', range=(g_min, g_max), bins=101,
             color='steelblue', edgecolor='black', lw=0.8, alpha=1, density=False)
#axes[1].hist(gamma_filtered_structures, label='Filtered Structures', range=(0, g_max), bins=101,
#             color='green', edgecolor='black', alpha=1, density=False)
axes[0].hist(gamma_selected_structures, label='Selected Structures', range=(g_min, g_max), bins=101,
             color='orange', edgecolor='black', lw=0.8, alpha=1, density=False)
#axes[0].hist(gamma_candidates, range=(0, gamma_max), bins=101,
#             color='green', edgecolor='black', alpha=1, density=True)
#plt.hist(get_gamma(descriptors, NEP.active), bins=50, color='orange', edgecolor='black')

axes[0].set_title('Distribution of Extrapolation Grade (gamma)')
axes[0].set_xlim(0, g_max)
axes[0].set_ylabel('Structures')
axes[0].legend()

axes[0].set_xlabel('Extrapolation Grade (gamma)')

#axes[0].grid(True, linestyle='--', alpha=0.5)
plt.show()

In [ ]:
def plot_active_set(A, idx):
    from sklearn.decomposition import PCA
    import matplotlib.pyplot as plt

    pca = PCA(n_components=2)
    A2 = pca.fit_transform(A)

    lf = LatexFigure()
    fig, axes = lf.create()
    # Plot ALL environments as a faint cloud
    axes[0].scatter(
        A2[:, 0], A2[:, 1],
        s=15,
        alpha=0.4,
        color="gray",
        label="All environments"
    )

    # Plot active set clearly
    axes[0].scatter(
        A2[idx, 0], A2[idx, 1],
        s=10,
        marker='x',
        #edgecolor="black",
        color="red",
        label="Active set"
    )
    axes[0].set_title("Active set coverage of descriptor space")
    axes[0].set_xlabel("PC1")
    axes[0].set_ylabel("PC2")
    axes[0].legend()
    #axes[0].tight_layout()
    plt.show()

In [ ]:
def check_active_set_quality(A, idx):
    import numpy as np
    import matplotlib.pyplot as plt

    sub = A[idx]
    inv = np.linalg.pinv(sub)

    # projection coefficients
    C = A @ inv  # (N, M)

    # reconstruction back in descriptor space
    A_proj = C @ sub  # (N, M)

    # relative error per atom
    residual = np.linalg.norm(A - A_proj, axis=1) / (np.linalg.norm(A, axis=1) + 1e-12)

    print(f"Mean reconstruction error: {np.mean(residual):.4e}")
    print(f"Max reconstruction error: {np.max(residual):.4e}")

    plt.hist(residual, bins=50)
    plt.title("Descriptor reconstruction error (active set quality)")
    plt.show()

In [ ]:
plot_active_set(NEP.descriptors, NEP.active_index)
check_active_set_quality(NEP.descriptors, NEP.active_index)

In [ ]:
#5. Run MD with the trained NEP model
md_params = dict(temperature=300,
                 n_steps=5000,
                 dump_interval=10)
NEP.setup_MD(**md_params)
#NEP.run_MD()       # run on HPC GPU node (surt.fysik.dtu.dk or sara.fysik.dtu.dk)

In [ ]:
from ase.visualize import view
md = read(os.path.join(NEP.iter_dir, 'md', "dump.xyz"), index=":")
view(md)

In [ ]:
# 6. Structure filtering based on gamma
NEP.filter_structures()

In [ ]:
# 7. Diversity selection with MaxVol
NEP.select_structures()

In [ ]:
from ase.io import read
from calorine.calculators import CPUNEP
from calorine.tools import relax_structure, get_force_constants

In [ ]:
from src.structure import Perovskite
perovskite = Perovskite('BaTiO3', N=2)
atoms = perovskite.atoms
calculator = CPUNEP('results/MLtest/iteration_1/nep.txt')
atoms.calc = calculator
relax_structure(atoms, fmax=0.00001)

In [ ]:
from ase import Atoms
from ase.build import bulk
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from ase.md.langevin import Langevin
from ase import units

MaxwellBoltzmannDistribution(atoms, temperature_K=300)

# 4. Set up MD (Langevin thermostat)
dyn = Langevin(
    atoms,
    timestep=1.0 * units.fs,
    temperature_K=300,
    friction=0.01
)

dyn.attach(lambda: write('results/MLtest/md.xyz', atoms, append=True), interval=10)
dyn.run(5000)

In [ ]:
from ase.visualize import view
view(read('results/MLtest/md.xyz', ':'))

read('MLtest/md.xyz')

In [ ]:
run_dir = "results/bulk/BaTiO3/0082/phonons"
structures = prepare_dataset(run_dir, n_rattle=100, n_strain=10)
#len(structures)

In [ ]:
# Load the phonon calculation results from the YAML file
phonon = phonopy.load("results/bulk/BaTiO3/0082/phonons/BaTiO3.yaml")
# Convert the phonon object to an ASE Atoms object
atoms = phonon_to_atoms(phonon, cell='super')
# Produce the force constants
phonon.produce_force_constants()
# Get the force constants matrix
fc = phonon.force_constants
# Generate new structures using the force constants and the original structure
structures_gen = generate_phonon_rattled_structures(atoms, fc2=fc, n_structures=100, temperature=300)

In [ ]:
from ase.visualize import view
view(atoms)

In [ ]:
import numpy as np

In [ ]:
# Calculate displacement vectors for each generated structure and print them
pos = atoms.get_positions()
for atoms_phonopy in phonon.supercells_with_displacements:
    ase_atoms = phonopy_to_ase(atoms_phonopy, bulk=True)
    pos_diff = ase_atoms.get_positions() - pos
    # Round the displacement vectors to 10 decimal places
    pos_diff = np.round(pos_diff, 10)
    # Take out the vector with the largest magnitude
    index = np.argmax(np.linalg.norm(pos_diff, axis=1))
    pos_diff = pos_diff[index]

    print(f"Atom {index}: {pos_diff}")
    
   

In [ ]:
phonon.forces

In [ ]:
fc_new = fc.transpose(0, 2, 1, 3)
fc_new = fc_new.reshape(3 * 40, 3 * 40)

In [ ]:
structures_gen = generate_phonon_rattled_structures(atoms, fc2=fc, n_structures=100, temperature=300)

In [ ]:
from ase.visualize import view
view(structures_gen)

In [ ]:
# Load numpy file
import numpy as np
np.load('results/bulk/SrTiO3/0001/frozen/R/mode_1/Q_1/time.npy')


In [ ]:
tests = read('results/MLtest/dataset.xyz', ':')
# Determine how many structures have calculators
count = 0
for atoms in tests:
    if atoms.calc is not None:
        count += 1
print(f"Number of structures with calculators: {count} out of {len(tests)}")

In [ ]:
len(tests)

In [ ]:
energies = {'Ba': -761.227747,
            'Ti': -1604.974503,
            'O': -440.177463}

with open(os.path.join(run_dir, 'energies.json'), 'w') as f:
    json.dump(energies, f)

In [ ]:
# Load the energies from the JSON file
with open(os.path.join(run_dir, 'energies.json'), 'r') as f:
    energies1 = json.load(f)

energies1

In [ ]:
unique_elements = set()
atom_energies = []
for atoms in structures:
    N_atoms = len(atoms)
    elements = atoms.get_chemical_symbols()
    unique_elements.update(elements)
    atoms.calc.results['energy'] -= sum(energies[element] for element in elements)
    #atoms.calc.results['energy'] /= N_atoms
    atom_energies.append(atoms.calc.results['energy'])

N_elements = len(unique_elements)

In [ ]:
structures[0].get_potential_energy()

In [ ]:
' '.join(unique_elements)

In [ ]:
len(structures)

In [ ]:
print(len(structures))
print(type(structures))

In [ ]:
# Shuffle order of structures
import random
random.shuffle(structures)

In [ ]:

# prepare input for NEP construction
parameters = dict(version=4,
                  type=[N_elements, ' '.join(unique_elements)],
                  cutoff=[8, 4],
                  neuron=30,
                  generation=100000,
                  batch=1000000)

setup_training(parameters, structures,
               rootdir='MLtest', overwrite=True,
               mode='kfold', n_splits=10)

In [ ]:
len(read('MLtest/nepmodel_split1/test.xyz@0:'))

## Model analysis

In [ ]:
import numpy as np
import pandas as pd
from ase.units import GPa
from calorine.nep import get_parity_data, read_loss, read_structures
from matplotlib import pyplot as plt
from pandas import DataFrame, concat as pd_concat
from sklearn.metrics import r2_score, mean_squared_error

In [ ]:
loss = read_loss('results/MLtest/nepmodel_split1/loss.out')

lf = LatexFigure()

#fig, axes = plt.subplots(figsize=(6.0, 4.0), nrows=2,sharex=True)
fig, axes = lf.create(AR=0.25, subplots=(2, 1), sharex=True)

ax = axes[0]
ax.set_ylabel('Loss')
ax.plot(loss.total_loss, label='total')
ax.plot(loss.L2, label='$l_2$')
ax.plot(loss.L1, label='$l_1$')
ax.set_yscale('log')
ax.legend()

ax = axes[1]
ax.plot(loss.RMSE_F_train, label='forces (eV/Å)')
ax.plot(loss.RMSE_V_train, label='virial (eV/atom)')
ax.plot(loss.RMSE_E_train, label='energy (eV/atom)')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Generation')
ax.set_ylabel('RMSE')
ax.legend()

#fig.subplots_adjust(hspace=0)
fig.set_constrained_layout_pads(hspace=0.0, h_pad=0.0)
plt.show()

In [ ]:
training_structures, _ = read_structures('results/MLtest/nepmodel_split1')

units = dict(
    energy='eV/atom',
    force='eV/Å',
    virial='eV/atom',
    stress='GPa',
)
# Make a 2x2 grid of parity plots for energy, force, virial, and stress
fig, axes = lf.create(width=0.8, AR=0.9, subplots=(2, 2))
kwargs = dict(alpha=0.2, s=0.3)
axes = axes.flatten()

# Loop over the properties and units, get the parity data, calculate R2 and RMSE, and plot the parity plots
for icol, (prop, unit) in enumerate(units.items()):
    df = get_parity_data(training_structures, prop, flatten=True)
    R2 = r2_score(df.target, df.predicted)
    rmse = np.sqrt(mean_squared_error(df.target, df.predicted))

    ax = axes[icol]
    ax.set_xlabel(f'Target {prop} ({unit})')
    ax.set_ylabel(f'Predicted ({unit})')
    # Plot line x = y for reference
    ax.plot([df.target.min(), df.target.max()], [df.target.min(), df.target.max()], 'k-', lw=0.8, alpha=0.2)
    ax.scatter(df.target, df.predicted, **kwargs)
    ax.set_aspect('equal')
    mod_unit = unit.replace('eV', 'meV').replace('GPa', 'MPa')
    ax.text(0.1, 0.75, f'{1e3*rmse:.1f} {mod_unit}\n' + '$R^2= $' + f' {R2:.5f}', transform=ax.transAxes)
fig.align_labels()
plt.show()

## Phonon dispersion

In [ ]:
from ase.io import read
from calorine.calculators import CPUNEP
from calorine.tools import relax_structure, get_force_constants

In [ ]:
from src.structure import Perovskite
perovskite = Perovskite('BaTiO3')
atoms = perovskite.atoms
calculator = CPUNEP('results/MLtest/nepmodel_split1/nep.txt')
atoms.calc = calculator
relax_structure(atoms, fmax=0.00001)

In [ ]:
atoms.cell

In [ ]:
phonon_ML = get_force_constants(atoms, calculator, [2, 2, 2])

In [ ]:
from src.phononcalc import get_phonon_dispersion, get_phonon_dos, get_phonon_pdos, order_labels
from src.frozenphonon import get_displacement, get_unstable_mode_groups, phonon_to_atoms
from src.plotsettings import PlotSettings
import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator
from ase.io import read, write
from ase.visualize import view
import numpy as np
import pandas as pd

In [ ]:
# Load the phonon calculation results from the YAML file
phonon_LCAO = phonopy.load("results/bulk/BaTiO3/0082/phonons/BaTiO3.yaml")

In [ ]:
def animate_frozen_phonons(phonon, n_points=101, deg=True,
                           dir='results/bulk/BaTiO3/ML/frozen'):
    """
    """
    # Get current working directory (cwd)
    #cwd = os.getcwd()
    # Unitcell and formula from phonon object
    unitcell = phonon_to_atoms(phonon, cell='unit')
    #formula = unitcell.symbols

    # Dictionary for q-points
    q_dict = {
        'G': [0.0, 0.0, 0.0],
        'X': [0.5, 0.0, 0.0],
        'R': [0.5, 0.5, 0.5],
        'M': [0.5, 0.5, 0.0],
    }

    dd_dict = {
        'G': 1/n_points,
        'X': 1.5/n_points,
        'R': 5/n_points,
        'M': 3/n_points,
    }

    # Loop over q-points and perform frozen phonon calculations for each q-point
    for qpoint in q_dict.keys():
        # Set the results directory for the current q-point
        dir_q = os.path.join(dir, qpoint)
        # Define coordinates and displacement distance for the current q-point
        q = q_dict[qpoint]
        dd = dd_dict[qpoint]
        #amp = dd_dict[qpoint]*n_points
        #dd = 10/n_points
    
        # Get groups of degenerate unstable modes at the given q-point
        groups, stable = get_unstable_mode_groups(phonon, q)

        if stable:
            print(f"No unstable modes found at {qpoint}. Animation will not be created.")
            return
            
        for g_id, group in enumerate(groups):

            modes = group["modes"]
            freq = group["frequency"]

            dir_group = os.path.join(dir_q, f"mode_{g_id+1}")

            os.makedirs(dir_group, exist_ok=True)
            # Save the frequency of the mode to a text file
            with open(os.path.join(dir_group, "freq.txt"), "w") as f:
                f.write(f"{freq:.6f} THz")

            if not deg:
                modes = [modes[0]] # Only consider the first mode if degenerate modes are not considered

            for mode_id, modevec in enumerate(modes):

                dir_mode = os.path.join(dir_group, f"Q_{mode_id+1}")

                os.makedirs(dir_mode, exist_ok=True)

                modevec_sc, supercell, supercell_matrix = get_displacement(unitcell, q, modevec)

                # Generate the supercell and get the mode vector for the supercell
                #modevec_sc, supercell, supercell_matrix = get_displacement(unitcell, q, modevec)
                # Determine the supercell size in each direction from the diagonal of the supercell matrix
                nx, ny, nz = supercell_matrix.diagonal().astype(int)
                ncells = nx*ny*nz
                

                calc = CPUNEP('results/MLtest/nepmodel_split1/nep.txt')

                supercell_disp = supercell.copy()
                ref_positions = supercell.positions.copy()
                supercell_disp.calc = calc

                amp = 0
                amplitudes = []
                energies = []
                images = []
                
                while True:
                    # Displace the atoms according to the mode vector by dd
                    supercell_disp.positions = ref_positions + amp * modevec_sc
                    
                    # Run the calculation
                    energy = supercell_disp.get_potential_energy()

                    # Scale energy by the number of unit cells in the supercell to get energy per unit cell
                    energy = energy / ncells
                    energies.append(energy)
                    # Append the supercell structure with displacements, forces and stresses to the list of images
                    img = supercell_disp.copy()
                    images.append(img)
                    # Append amplitude and update for the next iteration
                    amplitudes.append(amp)
                    amp += dd
                    tol = 50*1e-3 # Tolerance for stopping the loop based on energy increase (in eV)
                    # Stop the loop, if the energy has increased by more than the tolerance compared to the first point
                    if len(energies) > 1 and energies[-1] - energies[0] > tol:
                        break
                
                # Save the supercell structures with displacements as an xyz file
                write(os.path.join(dir_mode, 'structures.xyz'), images)

                # Save amplitudes and energies as a CSV file
                df = pd.DataFrame({
                    'Amplitude': amplitudes,
                    'Energy': energies
                })
                df.to_csv(os.path.join(dir_mode, 'energies.csv'), index=False)


In [ ]:
animate_frozen_phonons(phonon_ML)

In [ ]:
from scipy.optimize import curve_fit

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit


# Double well potentials
def U_harm(x, omega):
    return -1/2 * omega * x**2


def U_anharm(x, a, b):
    return -1/2 * a * x**2 + 1/4 * b * x**4


def plotDWs(formula, ids, bulk=True, Ncells=1, mirror=False):

    if isinstance(ids, str):
        ids = [ids]

    ytickmarks = np.arange(-10, 30, 5)

    # ---------- Structure path ----------
    if bulk:
        struc = f'bulk/{formula}'
    else:
        struc = f'slab/{formula}/{Ncells}uc'

    # ---------- Determine q-points ----------
    first_dir = os.path.join('results', struc, ids[0], 'frozen')

    q_points = [
        d for d in os.listdir(first_dir)
        if os.path.isdir(os.path.join(first_dir, d))
    ]

    N = len(q_points)

    # ---------- Colors ----------
    color_cycle = plt.rcParams['axes.prop_cycle'].by_key()['color']
    colors = {id_: color_cycle[i % len(color_cycle)] for i, id_ in enumerate(ids)}

    # ---------- Create figure ----------
    fig, axes = plt.subplots(1, N, figsize=(3*N, 5), sharey='col')

    if N == 1:
        axes = [axes]

    PlotSettings().set_size(fig)
    plt.subplots_adjust(wspace=0.08)

    # ---------- Helper functions ----------

    def load_dw_data(id_, q):
        """Load all DW data for a given calculation id and q-point."""
        dir_q = os.path.join('results', struc, id_, 'frozen', q)

        if not os.path.exists(dir_q):
            return []

        datasets = []

        for mode in os.listdir(dir_q):

            dir_mode = os.path.join(dir_q, mode)

            if not os.path.isdir(dir_mode):
                continue

            try:
                with open(os.path.join(dir_mode, 'freq.txt')) as f:
                    label = f.read().strip()
            except FileNotFoundError:
                label = mode

            for deg in os.listdir(dir_mode):

                dir_deg = os.path.join(dir_mode, deg)

                if not os.path.isdir(dir_deg):
                    continue

                df = pd.read_csv(os.path.join(dir_deg, 'energies.csv'))

                datasets.append((label, df))

        return datasets


    def get_fit(df):

        x = df['Amplitude'].to_numpy()
        y = df['Energy'].to_numpy() * 1000

        y0 = df[df['Amplitude'] == 0]['Energy'].iloc[0] * 1000
        y = y - y0

        xfit = np.linspace(0, max(x), 1000)

        popt, _ = curve_fit(U_anharm, x, y)

        yfit = U_anharm(xfit, *popt)

        return x, y, xfit, yfit


    def plot_curve(ax, x, y, xfit, yfit, color, label=None):

        if mirror:
            xfit = np.concatenate((-xfit[::-1], xfit))
            yfit = np.concatenate((yfit[::-1], yfit))

        #ax.plot(x, y, '.', markersize=4, color=color, label=label)
        ax.plot(xfit, yfit, '-', lw=1, color=color, label=label)

        ax.plot(0, 0, 'r.', markersize=4)

        dx = 0.5 if max(xfit) < 2 else 1
        xtickmarks = np.arange(0, max(xfit), dx)

        xticklabels = np.where(
            xtickmarks % 1 == 0,
            xtickmarks.astype(int).astype(str),
            xtickmarks.astype(str)
        )

        ax.set_xticks(xtickmarks, xticklabels)
        ax.set_yticks(ytickmarks, ytickmarks.astype(str))

        ax.set_xlim(min(xfit), max(xfit))
        ax.set_ylim(-20, 25)

        PlotSettings().set_style_ax(ax, gridlines=True)

    # ---------- Main plotting loop ----------

    for i, q in enumerate(q_points):

        ax = axes[i]

        if q == 'G':
            ax.set_title(r'$\Gamma$ point')
        else:
            ax.set_title(f'{q} point')

        for id_ in ids:

            datasets = load_dw_data(id_, q)

            for j, (label, df) in enumerate(datasets):

                x, y, xfit, yfit = get_fit(df)

                plot_curve(
                    ax,
                    x, y,
                    xfit, yfit,
                    colors[id_],
                    label=f'{id_}: {label}' if j == 0 else None
                )

        ax.legend(loc='upper center')

        ax.set_xlabel(f'$Q_{i+1}$ (amu$^{{1/2}}$Å)')

        if i != 0:
            ax.set_yticklabels([])

    axes[0].set_ylabel(r'$\Delta$ Energy (meV/u.c.)')

    axes[-1].tick_params(axis='y', labelright=True, labelleft=False)

    plt.show()

In [ ]:
plotDWs('BaTiO3', ['0082', 'ML'])

In [ ]:
def plot_dispersion(phonons, vals, width=1, bulk=True, pDOS=False):
    """Function to plot the phonon dispersion and DOS together.
    Parameters:
    - phonons: List of Phonopy objects containing phonon data.
    - vals: List of labels for each phonon object.
    - width: Width of the figure as a fraction of the total width.
    - pDOS: Whether to plot the projected density of states (PDOS) (default is True).
    - bulk: Boolean indicating if the system is bulk (True) or slab (False).
    Returns:
    - None. The function creates a plot of the phonon dispersion and DOS.
    """

    # Define tickmarks for the x- and y-axis
    E_tickmarks = np.arange(-10, 26, 5)
    # Convert tickmarks to strings with i for negative numbers
    E_ticklabels = []
    for tick in E_tickmarks:
        if tick < 0:
            E_ticklabels.append(f'{abs(tick)}i')
        else:
            E_ticklabels.append(f'{tick}')

    # Define tickmarks for the x- and y-axis
    dos_tickmarks = np.arange(0, 7, 1)

    colors = ["black", "blue", "red", "purple", "orange", "green"]

    # Make a simple figure where graphs are plotted
    fig, axes = plt.subplots(1, 2, figsize=(9.6, 5),
                             sharey='col', gridspec_kw={'width_ratios': [1, 0.4]})
    
    # Define two axes, one for the band structure and one for the DOS
    ax1, ax2 = axes
    
    PlotSettings().set_size(fig, width)
    plt.subplots_adjust(wspace=0.08)
    
    def _plot_disp(ax, phonon, val, col='k', vlines=True):
        # Extract phonon dispersion data
        (dist, X, freq, labels) = get_phonon_dispersion(phonon, bulk)
        dist = np.array(dist)
        dist /= dist[-1][-1]  # Normalize distances to the total length of the path
        X /= X[-1]  # Normalize high symmetry point locations to the total length of the path
        if vlines:
            # Plot vertical lines at symmetry points
            ax.vlines(X, E_tickmarks[0], E_tickmarks[-1], color='0.5', lw=0.8)
        # Plot dashed line at 0
        #ax.axhline(y=0, color='k', linestyle=':')
        # Determine the number of segments between symmetry points and the number of modes
        n_segments = len(freq)
        n_modes = freq[0].shape[1]
        # Loop over all segments and modes and plot everything
        for i in range(n_segments):
            for j in range(n_modes):
                if i == 0 and j == 0:
                    ax.plot(dist[i], freq[i][:, j], color=col, lw=1, label=f'{val}')
                else:
                    ax.plot(dist[i], freq[i][:, j], color=col, lw=1)
        # Set x- and y-ticks
        ax.set_xticks(X, labels)
        ax.set_yticks(E_tickmarks, E_ticklabels)
        # Set x- and y-limits
        ax.set_xlim(X[0], X[-1])
        ax.set_ylim(E_tickmarks[0], E_tickmarks[-1])

    def _plot_dos(ax, phonon, val='DOS', col='k'):
        # Extract total DOS data
        (dosx, dosy) = get_phonon_dos(phonon, bulk)
        # Plot total DOS
        ax.plot(dosx, dosy, lw=1, color=col, label=f'{val}')
        if pDOS:
            ax.fill_between(dosx, dosy, color='lightgray', alpha=0.5)

    def _plot_pdos(ax, phonon):
        atom_colors = {'Ba': 'tab:blue', 'Sr': 'tab:purple',
                       'Ti': 'tab:orange', 'O': 'tab:red'}
        # Extract PDOS data
        (pdosx, pdosy, symbols) = get_phonon_pdos(phonon, bulk)
        # Plot PDOS
        for i in range(pdosx.shape[0]):
            ax.plot(pdosx[i], pdosy, lw=1, color=atom_colors[symbols[i]], label=f'{symbols[i]}')
        # Get all handles and labels
        handles, labels = ax.get_legend_handles_labels()
        # Remove duplicates and sort for the legend
        sorted_handles, sorted_labels = order_labels(symbols, handles, labels)
        # Add legend with duplicates removed and sorted labels
        ax.legend(sorted_handles, sorted_labels, loc='best', fontsize=14)

    # Plot dashed line at Fermi level for both subplots
    ax1.axhline(y=0, color='k', linestyle=':', lw=0.8)
    ax2.axhline(y=0, color='k', linestyle=':', lw=0.8)
    for i, phonon in enumerate(phonons):
        # Plot phonon dispersion and total DOS for PW
        _plot_disp(ax1, phonon, val=vals[i], col=colors[i])
        _plot_dos(ax2, phonon, val=vals[i], col=colors[i])
        if pDOS:
            # Plot PDOS for PW
            _plot_pdos(ax2, phonon)

    # Set x- and y-label
    ax1.set_xlabel('k-points')
    ax1.set_ylabel('Frequency (THz)')
    # Add minor tickmarks to the y-axis
    ax1.yaxis.set_minor_locator(AutoMinorLocator())
    
    ax2.set_xlabel('DOS (1/THz)')
    ax2.legend(loc='upper right')
    # Force x- and y-ticks
    ax2.set_xticks(dos_tickmarks, dos_tickmarks)
    ax2.set_yticks(E_tickmarks, E_ticklabels)
    # Set limits to match
    ax2.set_xlim(dos_tickmarks[0], dos_tickmarks[-1])
    ax2.set_ylim(E_tickmarks[0], E_tickmarks[-1])
    # Hide y-tick labels
    ax2.set_yticklabels([])
    
    # Show figure
    plt.show()

In [ ]:
plot_dispersion([phonon_LCAO, phonon_ML], ['LCAO', 'ML'], width=1, bulk=True, pDOS=False)

## Testing